In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import matplotlib
matplotlib.use('macosx') # or tk, qt. makes render in interactive window, not in notebook.

In [3]:
import os
import copy
import pickle

from collections import defaultdict

import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import schnetpack as spk
from schnetpack.datasets import MD17
from schnetpack.datasets.md22 import MD22

from imlff.common.util import proj_path
from imlff.model.solvers import fixed_point_iteration, anderson
from imlff.model.util import init_iterating_schnet, adapt_apply_state_dict
from imlff.data.util import load_md17_data
from imlff.visualization.colors import adjust_saturation, adjust_color_saturation
from imlff.loading import load_model, load_model_from_wandb

from schnetpack.atomistic.distances import PairwiseDistances
from tqdm.notebook import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device='mps'

In [4]:
molecule_name, n_atoms, tag_deq = 'ethanol', 9, 'qcml-group/dev-v10/model-t61ulrkx:v840' # deq best performing so far
# molecule_name, n_atoms, tag_deq = 'azobenzene', 24, 'qcml-group/v7/model-a76tw6ap:v6179' # deq best performing so far
# molecule_name, n_atoms, tag_deq = 'cumulene1_test', 6, 'qcml-group/dev-v10/model-ua2elkv5:v164' # so3
nnpot = load_model_from_wandb(tag_deq)
f = nnpot.representation.f

~/imlff/models/qcml-group/dev-v10/model-t61ulrkx:v840/model.ckpt
Loaded model from ~/imlff/models/qcml-group/dev-v10/model-t61ulrkx:v840/model.ckpt with cpu device


/Users/johannes/miniconda3/envs/schnet/lib/python3.12/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.4.0
/Users/johannes/miniconda3/envs/schnet/lib/python3.12/site-packages/pytorch_lightning/utilities/parsing.py:208: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.


In [5]:
n_evaluations_1d = 31
n_evaluations_1d = 11
n_evaluations_1d = 51
x = y = np.linspace(-.5, .5, n_evaluations_1d)
X, Y = np.meshgrid(x, y)

X.shape, x.shape

n_evaluations = len(X.flatten())
n_evaluations, x

(2601,
 array([-0.5 , -0.48, -0.46, -0.44, -0.42, -0.4 , -0.38, -0.36, -0.34,
        -0.32, -0.3 , -0.28, -0.26, -0.24, -0.22, -0.2 , -0.18, -0.16,
        -0.14, -0.12, -0.1 , -0.08, -0.06, -0.04, -0.02,  0.  ,  0.02,
         0.04,  0.06,  0.08,  0.1 ,  0.12,  0.14,  0.16,  0.18,  0.2 ,
         0.22,  0.24,  0.26,  0.28,  0.3 ,  0.32,  0.34,  0.36,  0.38,
         0.4 ,  0.42,  0.44,  0.46,  0.48,  0.5 ]))

In [20]:
# find out which point is linearly extrapolated well

data_config = dict(
            # Dataset=MD17,
            num_train=1,
            num_val=1,
            molecule=molecule_name,
            batch_size=10,
            split_id=0,
            split_file_path='/tmp/foosplit'
        )
data_module = load_md17_data(**data_config)
data_module._val_dataset.subset_idx = list(range(1000))
# data_module._val_dataloader = None
# data_module.dataset = None
# data_module.val_batch_size = 10
# data_module.setup()

z_stars_list = []
for batch in tqdm(data_module.val_dataloader()):
    out = nnpot(batch)
    zs = out['z_star'].view((10, -1)).detach().cpu().numpy()
    z_stars_list.append(zs)

z_stars = np.concatenate(z_stars_list, axis=0)
del z_stars_list

Using split file: /tmp/foosplit
Using AllToAllNeighborList.
DATA PATH:
 ~/imlff/data/MD17/ethanol.db
Initializing with batch_size::: 10
Setting up transforms for the datamodule...


100%|██████████| 1/1 [00:00<00:00, 355.12it/s]

NON-TMP PATH:
 ~/imlff/data/MD17/ethanol.db


  0%|          | 0/100 [00:00<?, ?it/s]

In [21]:
warmstart_error = np.linalg.norm(2*z_stars[1:-1] - z_stars[:-2] - z_stars[2:], axis=1)
fixpoint_distance = np.linalg.norm(z_stars[:-2] - z_stars[2:], axis=1)
warmstart_error.shape, fixpoint_distance.shape

((998,), (998,))

In [22]:
best_extrapolated_idx = np.argmin(warmstart_error) + 2
best_extrapolated_idx

np.int64(212)

In [23]:
best_extrapolated_idx

np.int64(212)

In [24]:
best_extrapolated_idx = (np.argsort(warmstart_error) + 2)[0]
best_extrapolated_idx = (np.argsort(warmstart_error*10-fixpoint_distance) + 2)[10]
best_extrapolated_idx = (np.argsort(warmstart_error*10-fixpoint_distance) + 2)[4]
best_extrapolated_idx = (np.argsort(warmstart_error*10-fixpoint_distance*3) + 2)[2]

# ethanol within first 1000 indices
best_extrapolated_idx = (np.argsort(warmstart_error-fixpoint_distance) + 2)[200] # decent
best_extrapolated_idx = (np.argsort(warmstart_error-fixpoint_distance) + 2)[0]
best_extrapolated_idx = (np.argsort(warmstart_error-fixpoint_distance) + 2)[1]
best_extrapolated_idx = (np.argsort(warmstart_error-fixpoint_distance) + 2)[3]

best_extrapolated_idx = (np.argsort(-warmstart_error) + 2)[1]
best_extrapolated_idx = (np.argsort(-warmstart_error) + 2)[3]
# best_extrapolated_idx = (np.argsort(-warmstart_error) + 2)[5]
print(best_extrapolated_idx)

aa = fixpoint_distance
if 0:
    bb = warmstart_error
    plt.ylabel('warmstart error')
else:
    bb = warmstart_error / fixpoint_distance
    plt.ylabel('warmstart error / fixpoint diff')

plt.scatter(aa, bb)
plt.scatter(aa[best_extrapolated_idx-2], bb[best_extrapolated_idx-2], color='red')
plt.xlabel('fixpoint diff');


if False:
    plt.show()
else:
    plt.close(plt.gcf())

703


In [25]:
i_interest = 29632 # + 20000
i_interest = 7777
i_interest = best_extrapolated_idx - 1

idx_repeated =  []
idx_only_interest = []
for i in range(i_interest-1, i_interest+2):
    idx_repeated.extend([i]*n_evaluations)
    idx_only_interest.append(i)

In [26]:
data_config = dict(
            # Dataset=MD17,
            num_train=1,
            num_val=1,
            molecule=molecule_name,
            batch_size=n_evaluations,
            split_id=0,
            split_file_path='/tmp/foosplit'
        )
repeated_data_module = load_md17_data(**data_config)
repeated_data_module._val_dataset.subset_idx = idx_repeated
# repeated_data_module.val_idx = idx_repeated
# repeated_data_module._val_dataloader = None
# repeated_data_module.dataset = None
# repeated_data_module.val_batch_size = n_evaluations
# repeated_data_module.setup()

repeated_loader = repeated_data_module.val_dataloader()

Using split file: /tmp/foosplit
Using AllToAllNeighborList.
DATA PATH:
 ~/imlff/data/MD17/ethanol.db
Initializing with batch_size::: 2601
Setting up transforms for the datamodule...


100%|██████████| 1/1 [00:00<00:00, 277.00it/s]

NON-TMP PATH:
 ~/imlff/data/MD17/ethanol.db


In [28]:
nnpot.eval()

NeuralNetworkPotential(
  (postprocessors): ModuleList(
    (0): CastTo32()
    (1): AddOffsets()
  )
  (representation): AtomisticDEQ(
    (f): AtomisticRepresentationStackable(
      (radial_basis): GaussianRBF()
      (cutoff_fn): CosineCutoff()
      (embedding): Embedding(100, 128)
      (electronic_embeddings): ModuleList()
      (filter_net): Dense(
        in_features=50, out_features=384, bias=True
        (activation): Identity()
      )
      (interactions): ModuleList(
        (0): PaiNNInteraction(
          (interatomic_context_net): Sequential(
            (0): Dense(in_features=128, out_features=128, bias=True)
            (1): Dense(
              in_features=128, out_features=384, bias=True
              (activation): Identity()
            )
          )
        )
      )
      (mixing): ModuleList(
        (0): PaiNNMixing(
          (intraatomic_context_net): Sequential(
            (0): Dense(in_features=256, out_features=128, bias=True)
            (1): Dense(
   

In [29]:
only_interest_module = load_md17_data(**data_config)
only_interest_module.val_idx = idx_only_interest
only_interest_module._val_dataloader = None
only_interest_module.dataset = None

if False:
    only_interest_module.val_batch_size = len(idx_only_interest)
    only_interest_module.setup()
    batch_only_interest = next(iter(only_interest_module.val_dataloader()))
    batch_only_interest['forw_max_iter'] = batch_only_interest['back_max_iter'] = 10
    batch_only_interest['implicit_diff'] = True
    nnpot(batch_only_interest)
    z_stars = batch_only_interest['z_star'].detach()
    z_stars_org = z_stars = z_stars.view(3, n_atoms, 4, -1)
    
    Rs = batch_only_interest['_positions'].detach()
    Rs_org = Rs = Rs.view(3, -1)
    print(Rs.shape)
    
else:
    only_interest_module.val_batch_size = 1
    only_interest_module.setup()
    z_stars = []
    Rs = []
    zts_list = []
    forw_res_list = []
    prev_z_star = None
    z0 = a = b = None
    for i, batch in enumerate(only_interest_module.val_dataloader()):
        batch['forw_max_iter'] = batch['back_max_iter'] = 10
        batch['forw_tol'] = batch['back_tol'] = 1e-4
        batch['return_zts'] = True
        batch['implicit_diff'] = True
        batch['z0'] = z0
        nnpot(batch)
        z_star = batch['z_star'].detach()
        print(z_star.shape)
        z_stars.append(z_star)
        
        Rs.append(batch['_positions'].detach().view(1,-1))
        zts_list.append(batch['zts'])
        forw_res_list.append(batch['forw_res'])
        
        if i==0:
            z0 = a = z_star
        elif i==1:
            b = z_star
            z0 = 2*b-a
        elif i==2:
            break
            
        
    z_stars = torch.stack(z_stars, dim=0).detach()
    Rs = torch.cat(Rs, dim=0).detach()
    
z_star_interest = z_stars[1]

# assert torch.allclose(Rs_org, Rs)
# assert torch.allclose(z_stars_org, z_stars, atol=1e-4)

Using split file: /tmp/foosplit
Using AllToAllNeighborList.
DATA PATH:
 ~/imlff/data/MD17/ethanol.db
Initializing with batch_size::: 2601
Setting up transforms for the datamodule...


100%|██████████| 1/1 [00:00<00:00, 199.05it/s]

NON-TMP PATH:
 ~/imlff/data/MD17/ethanol.db
Setting up transforms for the datamodule...
torch.Size([9, 4, 128])
torch.Size([9, 4, 128])
torch.Size([9, 4, 128])


In [30]:
for forw_res, zts in zip(forw_res_list, zts_list):
    for i, fr in forw_res.items():
        zt = zts[i]
        print(i, round(fr.min().item(), 4), round(zt.sum().item(), 3) if zt is not None else 'None')
    print()

1 nan 3.545
2 0.3064 1.041
3 0.0962 1.895
4 0.0379 2.151
5 0.0163 2.238
6 0.0079 2.277
7 0.0035 2.294
8 0.0017 2.309
9 0.0009 2.316
10 0.0004 2.32

1 0.0393 2.649
2 0.0138 2.795
3 0.0064 2.819
4 0.0028 2.801
5 0.0013 2.792
6 0.0006 2.786
7 0.0003 2.784
8 0.0001 2.784
9 0.0001 2.784
10 0.0 2.784

1 0.0084 3.419
2 0.0048 3.452
3 0.0019 3.438
4 0.0009 3.434
5 0.0004 3.433
6 0.0002 3.433
7 0.0001 3.432
8 0.0 3.432



In [31]:
n_evaluations

2601

In [32]:
import numpy as np
from scipy.interpolate import CubicSpline


n_interpolations = n_evaluations

x = np.array([-1., 0., 1.])
y = np.array(Rs.detach().numpy())
cs = CubicSpline(x, y, axis=0)
interpolated_t = np.linspace(-1, 1.8, n_interpolations)
interpolated_t = np.concatenate((interpolated_t, np.array([-1., 0, 1])))
# unique, sort
interpolated_t = np.sort(np.unique(interpolated_t))[:n_interpolations]
# retrieve indices of -1, 0, 1
idx_m101 = np.searchsorted(interpolated_t, [-1, 0, 1])
interpolated_points = torch.Tensor(cs(interpolated_t))
print('distance of cubic interpolation to reference points:')
plt.plot((interpolated_points[None] - Rs[:, None]).norm(dim=-1).T)
interpolated_points = interpolated_points.view(n_interpolations*n_atoms, 3)
print(interpolated_points.shape)



if False:
    plt.show()
else:
    plt.close(plt.gcf())

distance of cubic interpolation to reference points:
torch.Size([23409, 3])


In [33]:
bat = next(iter(repeated_loader))
assert bat['_positions'].shape == interpolated_points.shape, f"{bat['_positions'].shape} != {interpolated_points.shape}"
bat['_positions'] = interpolated_points
bat = PairwiseDistances.forward(None, bat)
bat['forw_max_iter'] = 10
bat['back_max_iter'] = 1
bat['implicit_diff'] = True
nnpot(bat)
z_stars_interpolated = bat['z_star'].detach()

In [34]:
import torch

# Define the input tensor with size torch.Size([3, 6144])
points = torch.randn(3, 6144)

def get_plane_basis(points, normed=True):
    with torch.no_grad():
        # Calculate two vectors spanning the plane
        v1 = points[1] - points[0]
        v2 = points[2] - points[0]

        # Normalize the first vector to get an L2 norm unit vector
        v1_unit = v1 / torch.norm(v1)

        # Make the second vector orthogonal to the first using the Gram-Schmidt process
        v2_orthogonal = v2 - torch.dot(v2, v1_unit) * v1_unit
        v2_unit = v2_orthogonal / torch.norm(v2_orthogonal)

        if normed:
            return v1_unit, v2_unit
        else:
            return v1, v2_orthogonal

In [35]:
if False:
    data_path = proj_path(f'plots/fixpoint_trajectory/{molecule_name}_{tag_deq.replace("/", "_")}')
    # torch.save((Uz,Sz,Vz), f'{data_path}/UzSzVz.pt')
    Uz,Sz,Vz = torch.load(f'{data_path}/UzSzVz.pt')

In [36]:
order = [0, 1, 2]
order = [0, 2, 1]

scale = .4
scale = .75
scale = 1.9
scale = 1.
normed = True
# scale = .01

# scale = 2.
# normed = True

z_stars_flat = z_stars.view(3, -1)
vec1, vec2 = get_plane_basis(z_stars_flat[order], normed=normed)

z0, z1 = z_stars_flat[order[:2]]
scale = torch.norm(z0 - z1) * 2

offset1 = vec1.repeat(n_evaluations,1) * X.reshape(-1,1)
offset2 = vec2.repeat(n_evaluations,1) * Y.reshape(-1,1)
perturbation = scale * (offset1 + offset2).view(n_evaluations, n_atoms, z_star_interest.shape[1], -1)

z_star_repeated = z_star_interest.repeat(n_evaluations,1,1,1)
z_perturbed = (z_star_repeated + perturbation).detach()

z_outs = []

_, n_f, n_z = z_star_interest.shape

for b in repeated_loader:
#     break
# for _ in range(3):

    b = PairwiseDistances.forward(None, b)
    n_atoms_total = n_evaluations*n_atoms
    n_atoms_total = sum(b['_n_atoms'])
    
    z_perturbed_batched = z_perturbed.view(n_atoms_total, n_f, n_z)
    z_out_batched = f(b, z_perturbed_batched).detach()
    z_out = z_out_batched.view(n_evaluations, n_atoms, n_f, n_z)
    z_outs.append(z_out)

/var/folders/3h/t42_2k2d0tqf3gl1_tjslcbr0000gn/T/ipykernel_84906/817781861.py:20: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  offset1 = vec1.repeat(n_evaluations,1) * X.reshape(-1,1)
/var/folders/3h/t42_2k2d0tqf3gl1_tjslcbr0000gn/T/ipykernel_84906/817781861.py:21: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  offset2 = vec2.repeat(n_evaluations,1) * Y.reshape(-1,1)


In [37]:
import numpy as np

import matplotlib.pyplot as plt

# Calculate the min and max values for normalization
min_val = float('inf')
max_val = float('-inf')

for z_out in z_outs:
    diff_per_molecule = (z_out - z_perturbed).view(n_evaluations, -1).norm(dim=-1)
    min_val = min(min_val, diff_per_molecule.min().item())
    max_val = max(max_val, diff_per_molecule.max().item())
    
print(f"Min: {min_val:.3f}, Max: {max_val:.3f}")

# Plot each image with the same color scale
fig, axs = plt.subplots(1, len(z_outs), figsize=(15, 5))
for z_out, ax in zip(z_outs, axs):
    diff_per_molecule = (z_out - z_perturbed).view(n_evaluations, -1).norm(dim=-1)
    im = ax.imshow(diff_per_molecule.view(n_evaluations_1d, n_evaluations_1d), vmin=min_val, vmax=max_val)
    print(diff_per_molecule.min().item())

# fig.colorbar(im, ax=ax)
if False:
    plt.show()
else:
    plt.close(plt.gcf())

Min: 0.000, Max: 1.077
0.011219150249294787
9.99962196517176e-05
0.010981586559624135


In [38]:
aligns = []
vecs = torch.stack((vec1, vec2))
for z_out in z_outs:
    delta_per_molecule = (z_out - z_perturbed).view(n_evaluations, -1)
    align = vecs.to(torch.float64) @ delta_per_molecule.T
    align = align.view(2, n_evaluations_1d, n_evaluations_1d)
    aligns.append(align)    

delta_to_center = z_star_interest.repeat(n_evaluations, 1, 1) - z_stars_interpolated
delta_to_center = delta_to_center.view(n_evaluations, -1)
project = vecs @ delta_to_center.T

In [39]:
with torch.no_grad():
    zts_projected_list = []
    for zts in zts_list:
        zts_projected = {}
        for t, zt in zts.items():
            if zt is None:
                zt = torch.zeros(2)
            else:
                zt = vecs @ zt.view(-1).T
                # zt = zt.view(2, n_evaluations_1d, n_evaluations_1d)
            zts_projected[t] = zt
        zts_projected_list.append(zts_projected)

/var/folders/3h/t42_2k2d0tqf3gl1_tjslcbr0000gn/T/ipykernel_84906/133944037.py:9: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4483.)
  zt = vecs @ zt.view(-1).T


In [40]:
for forw_res, zts in zip(forw_res_list, zts_list):
    for i, fr in forw_res.items():
        zt = zts[i]
        print(i, round(fr.min().item(), 4), round(zt.sum().item(), 3) if zt is not None else 'None')
    print()

1 nan 3.545
2 0.3064 1.041
3 0.0962 1.895
4 0.0379 2.151
5 0.0163 2.238
6 0.0079 2.277
7 0.0035 2.294
8 0.0017 2.309
9 0.0009 2.316
10 0.0004 2.32

1 0.0393 2.649
2 0.0138 2.795
3 0.0064 2.819
4 0.0028 2.801
5 0.0013 2.792
6 0.0006 2.786
7 0.0003 2.784
8 0.0001 2.784
9 0.0001 2.784
10 0.0 2.784

1 0.0084 3.419
2 0.0048 3.452
3 0.0019 3.438
4 0.0009 3.434
5 0.0004 3.433
6 0.0002 3.433
7 0.0001 3.432
8 0.0 3.432



In [41]:
plot_path = proj_path(f'../imlff-paper/figures/')

In [ ]:
# Plot each image with the same color scale
import matplotlib as mpl

cmap = mpl.colormaps['RdYlGn']
zt_warmstart = None

project_xyz = np.concatenate((-project, interpolated_t[None]), axis=0)

a,b,c=project_xyz[:, idx_m101].T
correction = zts_projected_list[-1][max(zts_projected_list[-1])] - c[:2]

if True:
  fig, axs = plt.subplots(1, len(z_outs), figsize=(15, 5))
  for i, ax in enumerate(axs):
      ax.streamplot(X, Y, *aligns[i],
                  #   color=np.hypot(align1, align2),
                    cmap='viridis', density=1.5)
      
      if zt_warmstart is not None:
        print(11)
        ax.scatter(*zt_warmstart, color=cmap(0), s=200, zorder = 99)
        zt_corrected = [zt_warmstart]
      else:
        zt_corrected = []
      zts_projected = zts_projected_list[i]
      for k, zt in zts_projected.items():
        zt = (zt - correction) # 
        # zt = zt.numpy()[::-1] * -1 
        zt_corrected.append(zt)
        ax.scatter(*zt, color=cmap(k+1/10), zorder = 99)
        
      if zt_warmstart is None:
        zt_warmstart = zt
      else:
        zt_warmstart = 2*zt - zt_warmstart
        
      ax.plot(*np.array(zt_corrected).T, zorder = 10000)
      
      ax.set_title(f"Step {i+1}: {i/2:g}fs")
      ax.set_xlim(X.min(), X.max())
      ax.set_ylim(Y.min(), Y.max())
  # save
  pat = f'{plot_path}/streamplot_2D_V1_{n_evaluations_1d}.pdf'
  plt.savefig(pat)
  pat = f'{plot_path}/streamplot_2D_V1_{n_evaluations_1d}--edited.pdf'
  plt.savefig(pat)
  print(pat)

/var/folders/3h/t42_2k2d0tqf3gl1_tjslcbr0000gn/T/ipykernel_84906/2448817546.py:10: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  correction = zts_projected_list[-1][max(zts_projected_list[-1])] - c[:2]


11
11
~/imlff/../imlff-paper/figures//streamplot_2D_V1_51--edited.pdf


: 

In [28]:
n_fine = 100

alignment = vecs.to(torch.float64) @ perturbation.view(n_evaluations, -1).T
XX, YY = alignment.view(2, n_evaluations_1d, n_evaluations_1d)

xx, yy = XX[0], YY[:, 0]

# XX[:,:] = np.linspace(min(xx), max(xx), len(xx))[None, :]
# YY[:,:] = np.linspace(min(yy), max(yy), len(yy))[:, None]
xx = np.linspace(min(xx), max(xx), len(xx))
yy = np.linspace(min(yy), max(yy), len(yy))
xxx = np.linspace(min(xx), max(xx), n_fine)
yyy = np.linspace(min(yy), max(yy), n_fine)

XX, YY = np.meshgrid(xx, yy)
XXX, YYY = np.meshgrid(xxx, yyy)

In [29]:


def get_quiver_lines(
    quiver_density=.5,
    offsets = [-1, 0, 1],
    cmap = plt.cm.get_cmap('viridis'),  # Get the colormap
    show = False,
):

    lines_dict = defaultdict(list)
    for i, offset in enumerate(offsets):
        # Compute streamlines using quiver-like approach
        norms = aligns[i].norm(dim=0)
        norms = norms / norms.max()
        colors_mapped = cmap(norms)

        fig, ax = plt.subplots()
        res = ax.streamplot(XX, YY, *aligns[i], density=quiver_density, linewidth=.5, arrowstyle='-|>', arrowsize=3)
        lines_dict[offset] = res.lines.get_paths()
        if show:
            plt.show()
        else:
            fig.clear()
            
    return lines_dict

/var/folders/3h/t42_2k2d0tqf3gl1_tjslcbr0000gn/T/ipykernel_89638/568945869.py:4: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('viridis'),  # Get the colormap


In [ ]:
def upscale(arr):
    interp_spline = RectBivariateSpline(xx, yy, arr)
    return interp_spline(xxx, yyy)

In [242]:

from imlff.visualization.arrows_3d import arrows_3d
from imlff.visualization.arrows_3d_flat import arrows_3d_flat
from imlff.visualization.star_hexa_3d_flat import draw_hex_star

In [ ]:
warmstart_color = '#5750F6'
warmstart_color = '#2C287A' # dark blue
warmstart_color = 'black'
warmstart_color = 'orange'
warmstart_color = '#DE8E39'
# warmstart_color = "#E03BE0"
warmstart_color = "#671220"
warmstart_color = "#ECD872"
warmstart_color = "#D6F13C"
warmstart_color = "#519D00"
warmstart_color = "#9B2F2F"

# warmstart_color = 'red'
streamline_color = 'white'
# streamline_color = warmstart_color
streamline_color = '#4740C5' # dark blue
streamline_color = '#2C287A' # dark blue
# streamline_color = '#474095' # dark blue
streamline_color = '#474069' # dark blue

streamline_color = '#3E215D'
streamline_color = np.array([50/255, 33/255, 87/255])
streamline_color = '#2C287A' # dark blue
streamline_color = np.array([44/255, 40/255, 122/255])
streamline_color = plt.cm.get_cmap('Blues')(1.)
streamline_color = adjust_color_saturation(streamline_color, .3), adjust_color_saturation(streamline_color, .8), adjust_color_saturation(streamline_color, 1.)

colmaps = 'Purples', 'Blues', 'Greens'
base_cmap = 'Blues'
base_cmap = 'Blues_r'
colmaps = adjust_saturation(base_cmap, factor=.3), adjust_saturation(base_cmap, factor=.86), adjust_saturation(base_cmap, factor=1.5)
colmaps = 'Purples_r', 'Greens_r', 'Blues_r'
surface_alpha_by_strength = None
surface_color_offset = 0.
surface_color_scale = 1.5
surface_color_scale = 1.2

surface_color_power = .5
surface_color_scale = .8

surface_color_power = 1.5
surface_color_scale = 2
# surface_alpha_by_strength = .8

surface_alpha_by_strength = None
surface_color_power = 1.6
surface_color_scale = surface_color_power * 1.2

surface_alpha_by_strength = .3
surface_color_power = 2
surface_color_scale = surface_color_power * 1.2

surface_alpha_by_strength = .3
surface_color_power = 2.4
# surface_color_power = 2.
surface_color_scale = surface_color_power * 2.
surface_color_offset = 0

from imlff.visualization.colors import white_zero_cmap
P,G,B = white_zero_cmap('Purples', reverse=True,), white_zero_cmap('Greens', reverse=True,), white_zero_cmap('Blues', reverse=True)
P,G,B = 'Purples_r', 'Greens_r', 'Blues_r'
O = 'Oranges'
O = white_zero_cmap('Oranges', reverse=True,)
colmaps = O,O,O
colmaps = P,P,P
colmaps = G,G,G
colmaps = P,G,B

GR = 'Greys_r'
# base_cmap = 'Oranges'
colmaps = adjust_saturation(GR, factor=.04), adjust_saturation(GR, factor=.7), adjust_saturation(GR, factor=1.4)

quiver_density = .6
quiver_density = .32
quiver_density = .8
quiver_density = .5
lines_dict = get_quiver_lines(quiver_density)
plt.close('all')


quiver = 0.01
quiver = -0.00001
surface = False

quiver = False
surface = True

colmaps = GR,GR,GR
colmaps = B,B,B

surface_color_offset = -.05

surface_color_offset = 0
surface_color_scale = .9
# surface_color_scale = 1.

# # more uniform
# surface_color_offset = .1
# surface_color_scale = .7

# # less uniform
# surface_color_offset = -.03
# surface_color_scale = 1.


streamline_color = [plt.cm.get_cmap('Blues')(.1)] * 3
streamline_color = [plt.cm.get_cmap(cmap)(.7) for cmap in colmaps]
frame = 0.0

streamline_color = [plt.cm.get_cmap(cmap)(.8) for cmap in colmaps]
frame = 0.0

ht_color = '#1f77b4' # dark blue
ht_color = '#440154' # low end of viridis
ht_color = "#353535"
ht_color = "#0A0A0A"
# ht_color = 'grey'

zts_colour = '#2C287A' # dark blue
zts_colour = "#1A175E" # darker blue
# zts_colour = "#000000" # black
# zts_colour = "#7B1B31"

arrows_3d_zt = arrows_3d_flat
# arrows_3d_zt = arrows_3d

zts_kwargs = dict(
    # arrowstyle='-',
    arrowstyle='-|>',
    mutation_scale=15, 
    color=zts_colour,
    linewidth=2,
    shrinkA=3,
    shrinkB=2,
    
    head_length=0.06,
    shaft_width=0.018,
)
zts_min_diff = 0.05 * 0.4

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d', computed_zorder=False)

# show subset of panes
clip = lambda arr: arr[:40, :]
clip = lambda arr: arr
# frame = False

# boundaries in x,y direction
n, s = clip(XX).min(), clip(XX).max()
w, e = clip(YY).min(), clip(YY).max()

offsets = [-1, 0, 1]
if surface: # surface plot
    XX_round = clip(XX).copy()
    YY_round = clip(YY).copy()
    
    if False: # round corners (not working)
        r = 0.1  # corner radius
        mask = (np.abs(XX) > 0.95 - r) & (np.abs(YY) > 0.95 - r)
        d = np.sqrt((np.abs(XX[mask]) - (0.95 - r))**2 + (np.abs(YY[mask]) - (0.95 - r))**2)
        scale = np.clip((r - d) / r, 0, 1)
        XX_round[mask] *= scale
        YY_round[mask] *= scale
            
    for i, offset in enumerate(offsets):
        colmap = colmaps[i] if type(colmaps) is tuple else colmaps
        zorder = 10000*offset
        Z = np.full_like(X, offset)  # Set Z position for each layer
        if True:  # filled contour regions (discrete color bands)
            norms = aligns[i].norm(dim=0)
            # norms = norms**surface_color_power / norms.max()**surface_color_power * surface_color_scale 
            norms_np = clip(norms.numpy())

            levels = np.linspace(norms_np.min(), norms_np.max(), 12)

            if surface_alpha_by_strength is not None:
                # Build per-level alpha: stronger regions more transparent
                cmap_base = plt.cm.get_cmap(colmap)
                norm_obj = plt.Normalize(vmin=norms_np.min(), vmax=norms_np.max())
                level_mids = 0.5 * (levels[:-1] + levels[1:])  # midpoint of each band
                colors = []
                for mid in level_mids:
                    mid_normed = norm_obj(mid)
                    mid_normed = max(0, mid_normed + surface_color_offset) / (1 + surface_color_offset) * surface_color_scale
                    rgba = list(cmap_base(mid_normed))
                    strength = mid_normed ** surface_alpha_by_strength
                    rgba[3] = 1.0 - strength  # same alpha logic as surface
                    colors.append(rgba)
                ax.contourf(XX_round, YY_round, norms_np,
                            levels=levels,
                            colors=colors,
                            zdir='z', offset=offset,
                            zorder=zorder)
            else:
                ax.contourf(XX_round, YY_round, norms_np,
                            levels=levels,
                            cmap=colmap,
                            alpha=0.7,
                            zdir='z', offset=offset,
                            zorder=zorder)
        if False: # surface plot for colours
            norms = aligns[i].norm(dim=0)
            norms = norms**surface_color_power / norms.max()**surface_color_power * surface_color_scale - surface_color_offset
            cmap = plt.cm.get_cmap(colmap)  # Get the colormap
            cols = cmap(norms)
            
            if surface_alpha_by_strength is not None:
                lengths = (norms / norms.max()).numpy() ** surface_alpha_by_strength
                cols[:,:,-1] *= 1. - lengths # sets alpha smaller for large lengths
            
            ax.plot_surface(XX_round, YY_round, Z, facecolors=cols, 
                            alpha=.7 if surface_alpha_by_strength is None else None,
                            rstride=1, cstride=1, linewidth=0, antialiased=True, shade=False, zorder=zorder)
        if False:  # contour lines instead of surface
            norms = aligns[i].norm(dim=0)
            # norms = norms**surface_color_power / norms.max()**surface_color_power * surface_color_scale - surface_color_offset
            norms_np = norms.numpy()

            cmap = plt.cm.get_cmap(colmap)
            levels = np.linspace(norms_np.min(), norms_np.max(), 12)

            # Draw contour lines at fixed Z height
            contour = ax.contour(XX_round, YY_round, norms_np,
                                 levels=levels,
                                 zdir='z', offset=offset,
                                 cmap=colmap,
                                 linewidths=1.2,
                                 alpha=0.4,
                                 zorder=zorder,
                                 )
        if False:
            # Compute streamlines using quiver-like approach
            x, y = XX, YY
            norms = aligns[i].norm(dim=0)
            norms = norms / norms.max()
            x, y, norms = XXX, YYY, upscale(norms)
            cmap = plt.cm.get_cmap(colmap)  # Get the colormap
            cols = cmap(norms)#.reshape(-1, 4)
            
            cols = gaussian_filter(cols, sigma=1)
            
            ax.plot_surface(x, y, np.full_like(x, offset)-.05, facecolors=colors_mapped, alpha=0.3, rstride=1, cstride=1)
        if False:
            ax.quiver(XX, YY, Z, *aligns[i], np.zeros_like(X), length=0.1, color=colors_mapped, alpha=0.8)
            ax.quiver(X, Y, Z, *aligns[i], np.zeros_like(X), length=0.1, color=colors_mapped, alpha=0.8)
            ax.contourf(XX, YY, Z, colors=colors_mapped, alpha=0.8)
            min_coords = np.unravel_index(norms.argmin().item(), norms.shape)
            coords = XX[min_coords], YY[min_coords]
            ax.scatter(*coords, offset, color='blue', s=100)   
if True: # quiver, frame
    for i, (offset, lines) in enumerate(lines_dict.items()):
        quiver_color = mcolors.to_rgba(streamline_color[i] if type(streamline_color) in (list, tuple) else streamline_color)
        if quiver is not False:
            quiver_kwargs = dict(
                color=quiver_color, 
                zorder=offset*10000 + 10 * np.sign(quiver),
                alpha=.9, 
                linewidth=.9, 
            )
            for line in lines:
                line = line.vertices
                ax.plot(line[:, 0], line[:, 1], offset+quiver, **quiver_kwargs)
          
        if frame is not False:
            z_frame = offset+frame # scalar
            # ax.plot([n,w], [n,e], z_frame, **quiver_kwargs)
            # ax.plot([n,e], [n,w], z_frame, **quiver_kwargs)
            # ax.plot([s,w], [s,e], z_frame, **quiver_kwargs)
            # ax.plot([s,e], [s,w], z_frame, **quiver_kwargs)
            
            fill_factor, edge_factor = 1.05, 1. # smaller is darker
            fill_color = tuple(np.clip(c * fill_factor, 0, 1) for c in quiver_color[:3]) + (quiver_color[3],)
            edge_color = tuple(np.clip(c * edge_factor, 0, 1) for c in quiver_color[:3]) + (quiver_color[3],)

            surf_kw = dict(
                alpha=.9, 
                linewidth=0.4,
                antialiased=False,
                color=fill_color,
                edgecolor=edge_color,
                shade=False,
            )

            frame_height = 0.04
            
            for (x1, y1), (x2, y2), back in [
                # ((n, n), (w, e), False),
                # ((n, n), (e, w), True),
                # ((s, s), (w, e), False),
                # ((s, s), (e, w), True),
                ((n, w), (n, e), False),
                ((n, w), (s, w), True),
                ((s, w), (s, e), True),
                ((n, e), (s, e), False),
            ]:
                
                kw = surf_kw.copy()
                zorder = offset*10000
                if back:
                    zorder -= 20
                    dz = 0
                    kw |= dict(color='white')
                else:
                    zorder += 20
                    dz = frame_height
                    
                xx = np.array([[x1, x2], [x1, x2]])
                yy = np.array([[y1, y2], [y1, y2]])
                zz = np.array([[z_frame     , z_frame     ],
                               [z_frame - dz, z_frame - dz]])
                    
                ax.plot_surface(xx, yy, zz, zorder=zorder, **kw)
        
project_xyz = np.concatenate((-project, interpolated_t[None]), axis=0)
if True: # stars
    def draw_star_flat(ax, x, y, z, color, size=0.05, n_arms=6):
        """Draw a LaTeX-style * lying flat in the z-plane."""
        angles = np.linspace(0, 2*np.pi, n_arms, endpoint=False)
        for angle in angles:
            dx = size * np.cos(angle)
            dy = size * np.sin(angle)
            ax.plot([x - dx, x + dx], [y - dy, y + dy], [z, z],
                    color=color, linewidth=1.5, zorder=zorder)
            
            
    for ti, zorder in zip(idx_m101, [-5000, 5000, 15000]):
        coord = project_xyz[:, ti]
        # ax.scatter(x,y,z, color=ht_color, s=100, zorder=zorder)
        # draw_star_flat(ax, *coord, color=ht_color, size=0.05)
        draw_hex_star(ax, coord, color=ht_color, scale=0.06, weight=.9, zorder=zorder)
if True: # h*(t) arrow
    end_at = -300
    for l,u,zorder in [
        (idx_m101[0], idx_m101[1], -5000),
        (idx_m101[1], idx_m101[2],  5000),
        (idx_m101[2],      end_at, 15000),
    ]:
        ax.plot(*project_xyz[:, l:u], color=ht_color, linewidth=2, zorder=zorder)
        
    prev = 100
    prev = 5
    starts = np.array([*-project[:, end_at-prev], interpolated_t[end_at-prev]])
    ends   = np.array([*-project[:, end_at     ], interpolated_t[end_at     ]])
    diff = starts-ends
    starts -= diff*4
    ends   -= diff*4
    arrows_3d(starts=starts, ends=ends, ax=ax, mutation_scale=30, arrowstyle='-|>', linewidth=0, color=ht_color, shrinkA=0, shrinkB=0, zorder=1e9)
if False: # labels
    for t in range(3):
        ax.text(.7, -1, t-.7, r'$t_'+str(t+1)+' = '+str(t)+r'fs$'+'\n'+r'$\frac{\delta\mathbf{h}}{\delta\tau}|_{t_'+str(t+1)+r'}$', color='black', fontsize=12)
    x,y,z=starts[0]
    x+=.1
    z-=.6
    # ax.text(x,y,z, r'$\frac{\delta \mathbf{h^*}}{\delta t}$', color=ht_color, fontsize=22)
    ax.text(x,y,z, r'$\mathbf{h^*}(t)$', color=ht_color, fontsize=22)
if True: # warmstarts
    a,b,c=project_xyz[:, idx_m101].T
    C = 2*b-a # prediction for c
    a_up = a.copy(); a_up[2] = b[2]
    starts_const, ends_const = a, a_up
    starts_lin1, ends_lin1 = a, (b + 0.*(b-a)) # we manually extend the end by 0.15 too far, bc the rendering stops too early
    
    way = C-b
    starts_lin2_tip, ends_lin2_tip = b+.8*way, C
    starts_lin2_arr, ends_lin2_arr = b, C-.1*way
    starts_lin2_tip, ends_lin2_tip = b, C
    starts_lin2_arr, ends_lin2_arr = b, C
    kwd = dict(
        ax=ax,
        
        color=warmstart_color,
        
        shrinkA=5,
        shrinkB=10,
        
        # linewidth=4, # thin
        linewidth=0,
        capstyle='round',  # rounded ends
        joinstyle='miter',
        
        mutation_scale=30, # arrow tip size
        arrowstyle='-|>',
    )
    arrows_3d(starts=starts_const, ends=ends_const, **kwd, zorder=-10000+100)
    arrows_3d(starts=starts_lin2_tip, ends=ends_lin2_tip, **kwd, zorder=100)
    kwd.update(dict(
        arrowstyle = '-',
        shrinkB = 0,
        # dash pattern:
        linestyle=(0, (1.5, 2)), # dashed
        linewidth=5, # thick
    ))
    arrows_3d(starts=starts_const, ends=ends_const, **kwd, zorder=-10000+100)
    arrows_3d(starts=starts_lin1, ends=ends_lin1, **kwd, zorder=-10000+100)
    arrows_3d(starts=starts_lin2_arr, ends=ends_lin2_arr, **kwd, zorder=100)
        
    # # at middle of thick arrow print text
    # middle_a, middle_b = (starts_thick + ends_thick) / 2
    # middle_a += np.array([-.9, 0, 0])
    # middle_b += np.array([.1, 0, 0])
    # ax.text(*middle_a, 'constant warmstart', color=warmstart_color, fontsize=12)
    # ax.text(*middle_b, 'linear warmstart', color=warmstart_color, fontsize=12)
if True: # zts
    correction = zts_projected_list[-1][max(zts_projected_list[-1])] - c[:2]
    starts, ends = [], []
    for offset, forw_res, zts_projected, z_star_projected in zip(offsets, forw_res_list, zts_projected_list, project_xyz[:2, idx_m101].T):
        prev_zt = None
        counter = 0
        # prev_zt = np.array([1., -1.])
        
        if False:
            for i, zt in zts_projected.items():
                if (zt==0).all():
                    zt = np.array([XX.max()-0.25, YY.min()])
                    zt = np.array([XX.max(), YY.min()])
                else:
                    # adapt to coordinate axes
                    zt = (zt - correction) * -1 
                fr = forw_res.get(i, np.nan)
                if i==0:
                    prev_zt = zt
                # elif (np.isnan(fr.min()) or fr.max()>.02):
                elif np.linalg.norm(zt - prev_zt) > zts_min_diff:
                    starts.append([*-prev_zt, offset])
                    ends  .append([*-zt,      offset])
                    # starts = starts_lin2
                    # ends = ends_lin2
                    
                    prev_zt = zt     
                    counter += 1         
                    
        else:
            def get_closest_corner(zt):
                closest_distance = np.inf
                for corner in [[n,-e], [n,-w], [s,-e], [s,-w]]:
                    corner = torch.tensor(corner)
                    distance = (corner-zt).norm()
                    if distance < closest_distance:
                        closest_corner = corner
                        closest_distance = distance
                return closest_corner
            prev_zt = None
            counter = 0
                        
            for i, zt in zts_projected.items():
                if (zt == 0).all():
                    prev_zt = None
                else:

                    # adapt to coordinate axes
                    zt = (zt - correction) * -1 
                        
                    if i==0:
                        prev_zt = zt
                        
                    else:
                        if prev_zt is None:
                            prev_zt = get_closest_corner(zt)
                            
                        # if (np.isnan(fr.min()) or fr.max()>.02):
                        if np.linalg.norm(zt - prev_zt) > zts_min_diff:
                            starts.append([*-prev_zt, offset])
                            ends  .append([     *-zt, offset])
                            
                            prev_zt = zt     
                            counter += 1             
            
        # print(i, counter, len(ends))
        if counter:
            ends[-1][0], ends[-1][1] = z_star_projected
        else:
            print('Warning: No arrows in layer', offset+2)
            
    zts_kwargs['ax'] = ax
    
    # extract special first arrow: split it into four arrows to make it appear dashed.
    start_1, *starts = np.array(starts)
    end_4, *ends = np.array(ends)
    
    zts_kwargs['shrinkA'] = 8
    arrows_3d_zt(starts=np.array(starts), ends=np.array(ends), **zts_kwargs, zorder=50000)

    if True:
        # calculate four subarrows:
        end_2 = start_3 = end_4 / 2     + start_1 / 2
        end_1 = start_2 = end_2 / 4 * 3 + start_1 / 4
        end_3 = start_4 = end_4 / 4     + start_3 / 4 * 3
        starts_wo_tips = np.array([start_1, start_2, start_3])
        ends_wo_tips   = np.array([end_1,   end_2,   end_3])
        starts_w_tips = np.array([start_4])
        ends_w_tips   = np.array([end_4])
    else:
        # three segments
        end_3 = end_4
        end_1 = start_2 = end_3 / 3     + start_1 / 3 * 2
        end_2 = start_3 = end_3 / 3 * 2 + start_1 / 3
        starts_wo_tips = np.array([start_1, start_2, ])
        ends_wo_tips   = np.array([end_1,   end_2,   ])
        starts_w_tips = np.array([start_3])
        ends_w_tips   = np.array([end_3])
    
    zts_kwargs['shrinkA'] = 20
    zts_kwargs['shrinkB'] = 8
    arrows_3d_zt(starts=starts_w_tips, ends=ends_w_tips, **zts_kwargs, zorder=50000)
    # remove tip:
    zts_kwargs['shrinkB'] = 5
    # zts_kwargs['shrinkB'] = 8
    zts_kwargs['arrowstyle'] = '-'
    arrows_3d_zt(starts=starts_wo_tips, ends=ends_wo_tips, **zts_kwargs, zorder=50000)
if True: # rendering
    ax.set_aspect('equalxy')  # options: 'equal', 'equalxy', 'equalxz', 'equalyz'
    # Set viewing angle
    ax.view_init(elev=10, azim=230)
    ax.view_init(elev=19, azim=230)
    ax.view_init(elev=14, azim=243)
    # ax.view_init(elev=12, azim=80)
    # ax.view_init(elev=26, azim=230)
    ax.invert_yaxis()
    ax.set_proj_type('ortho')
    ax.set_axis_off()
if True: # saving
    plt.tight_layout()
    
    file_path = f'{plot_path}/streamplot_3D_V5_{n_evaluations_1d}'
    plt.savefig(f'{file_path}.svg')
    plt.savefig(f'{file_path}.pdf', transparent=True)
    print(f'{file_path}.pdf')
    
    plt.show()
    

/var/folders/3h/t42_2k2d0tqf3gl1_tjslcbr0000gn/T/ipykernel_89638/470572204.py:25: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  streamline_color = plt.cm.get_cmap('Blues')(1.)
/var/folders/3h/t42_2k2d0tqf3gl1_tjslcbr0000gn/T/ipykernel_89638/470572204.py:106: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  streamline_color = [plt.cm.get_cmap('Blues')(.1)] * 3
/var/folders/3h/t42_2k2d0tqf3gl1_tjslcbr0000gn/T/ipykernel_89638/470572204.py:107: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.ge

[ 0.44694201 -0.13311664  1.49684615]
[-0.32638574 -0.06213424 -0.1       ]
[0.29404881 0.05618819 0.9       ]
[-0.32638574 -0.06213424 -0.1       ]
[-0.03249569 -0.00608677 -0.1       ]
[0.29404881 0.05618819 0.9       ]
~/imlff/../imlff-paper/figures//streamplot_3D_V5_51.pdf


: 

In [88]:
n,s,w,e

(np.float64(-0.6339923284881244),
 np.float64(0.6339932056739884),
 np.float64(-0.6339923132458407),
 np.float64(0.35503637962264567))